<a href="https://colab.research.google.com/github/PauloRadatz/opendss-python-examples/blob/main/presentations/ieee_et_pes_pels_joint_chapter_workshop/04_introduction_to_py_dss_toolkit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4 — Introduction to py-dss-toolkit

In Notebooks 1 to 3 we used only `py-dss-interface`. That package is the bridge between Python and OpenDSS — it lets us send any OpenDSS command from Python and read results back. Everything we did was possible with just that.

But you probably noticed a pattern: a lot of the code we wrote was about *organizing* the results — pairing voltages with node names, building DataFrames, plotting. That work shows up in every study. So I built `py-dss-toolkit` to handle the common pieces.

Think of it this way: **`py-dss-interface` is how Python talks to OpenDSS. `py-dss-toolkit` builds on top of it to make common workflows shorter, cleaner, and more visual.**

In this notebook we redo parts of the previous workflows using the toolkit and compare the result.

## Setup

Installing `py-dss-toolkit` automatically pulls `py-dss-interface` as a dependency.

In [15]:
!pip install py-dss-toolkit
!git clone https://github.com/PauloRadatz/opendss-python-examples.git
%cd opendss-python-examples

fatal: destination path 'opendss-python-examples' already exists and is not an empty directory.
/content/opendss-python-examples/feeder_models/IEEETestCases/123Bus/opendss-python-examples


In [16]:
import py_dss_interface
from py_dss_toolkit import dss_tools

dss_file= "/content/opendss-python-examples/feeder_models/IEEETestCases/123Bus/IEEE123Master.dss"

print(f"Master file: {dss_file}")

Master file: /content/opendss-python-examples/feeder_models/IEEETestCases/123Bus/IEEE123Master.dss


## Link the DSS instance to the toolkit

We still create a normal `py_dss_interface.DSS()` object — that is unchanged. We then connect it to `dss_tools` with `update_dss(dss)`. From now on we have two ways to control OpenDSS: directly through `dss`, or through the higher-level helpers in `dss_tools`.

In [17]:
dss = py_dss_interface.DSS()
dss_tools.update_dss(dss)

print(f"OpenDSS started: {dss.started}")

OpenDSS started: True


## Compile and solve

Same as before. The toolkit does not replace the OpenDSS engine; it sits on top of it. However, some functionalities are replicated within the toolkit to streamline operations and leverage code completion features. For example, in thi case, you can use:

`dss_tools.configuration.compile_dss(dss_file)`

`dss_tools.simulation.solve_snapshot()`

In [18]:
dss.text(f"compile [{dss_file}]")
dss.text("solve")

''

## Access the model as DataFrames

`dss_tools.model` exposes structured DataFrames for the elements in the circuit. No more manual loops to collect properties — the toolkit does it for us.

Below we look at lines, buses, and a model summary. The `buses` DataFrame in particular is hard to assemble in standalone OpenDSS or with `py-dss-interface` alone — every bus, every property, in one table.

In [19]:
lines_df = dss_tools.model.lines_df
print(f"Lines in the circuit: {len(lines_df)}")
lines_df.head()

Lines in the circuit: 126


,name,bus1,bus2,linecode,length,phases,r1,x1,r0,x0,...,heightunit,conductors,normamps,emergamps,faultrate,pctperm,repair,basefreq,enabled,like
0,l115,149,1,1,0.4,3,----,----,----,----,...,m,,400.000000,600.000000,0.1,20,3,60,true,
1,l1,1.2,2.2,10,0.175,1,----,----,----,----,...,m,,400.000000,600.000000,0.1,20,3,60,true,
2,l2,1.3,3.3,11,0.25,1,----,----,----,----,...,m,,400.000000,600.000000,0.1,20,3,60,true,
3,l3,1.1.2.3,7.1.2.3,1,0.3,3,----,----,----,----,...,m,,400.000000,600.000000,0.1,20,3,60,true,
4,l4,3.3,4.3,11,0.2,1,----,----,----,----,...,m,,400.000000,600.000000,0.1,20,3,60,true,


In [20]:
buses_df = dss_tools.model.buses_df
print(f"Buses in the circuit: {len(buses_df)}")
buses_df.head()

Buses in the circuit: 132


,name,nodes,num_nodes,kv_base,distance,coord_defined,x,y,latitude,longitude,all_pce_active_bus,all_pde_active_bus,line_list,line_total_miles,load_list,section_id,total_customers
0,150,"[1, 2, 3]",3,2.401777,0.0,0,0.0,0.0,0.0,0.0,[Vsource.source],[Transformer.reg1a],[None],0.0,[None],0,0
1,150r,"[1, 2, 3]",3,2.401777,0.0,0,0.0,0.0,0.0,0.0,[None],"[Line.sw1, Transformer.reg1a]",[LINE.sw1],0.0,[None],0,0
2,149,"[1, 2, 3]",3,2.401777,0.0,0,0.0,0.0,0.0,0.0,[None],"[Line.l115, Line.sw1]","[LINE.l115, LINE.sw1]",0.0,[None],0,0
3,1,"[1, 2, 3]",3,2.401777,0.0,0,0.0,0.0,0.0,0.0,[Load.s1a],"[Line.l115, Line.l1, Line.l2, Line.l3]","[LINE.l115, LINE.l1, LINE.l2, LINE.l3]",0.0,[LOAD.s1a],0,0
4,2,[2],1,2.401777,0.0,0,0.0,0.0,0.0,0.0,[Load.s2b],[Line.l1],[LINE.l1],0.0,[LOAD.s2b],0,0


`dss_tools.model.summary_df` gives a quick model summary — number of buses, lines, transformers, loads, and so on. Useful as a sanity check after compiling.

In [21]:
model_summary = dss_tools.model.summary_df
model_summary

,count
buses,132.000000
nodes,278.000000
ckt elements,237.000000
vsource,1.000000
transformer,8.000000
regcontrol,7.000000
line,126.000000
capacitor,4.000000
load,91.000000
line length,38.983000


## Get snapshot results in one line

`dss_tools.results.summary_df` returns the most useful snapshot quantities — feederhead P and Q, total losses, min and max voltage — already organized in a small table.

In [22]:
results_summary = dss_tools.results.summary_df
results_summary

,Results
P feeder (kW),3615.241896
Q feeder (kvar),1311.510281
P losses (kW),95.976707
Q losses (kvar),192.499357
max voltage (pu),1.049961
min voltage (pu),0.979211


## Find the lowest voltage bus — toolkit version

Compare this with what we wrote in Notebook 3. The work is the same, but we are now reading an already-organized DataFrame.

`dss_tools.results.voltage_ln_nodes` returns a tuple of two DataFrames — magnitudes (in pu) and angles (in degrees). Each row is a bus, and the columns are `node1`, `node2`, `node3`.

In [23]:
vmag_df, vang_df = dss_tools.results.voltage_ln_nodes
vmag_df.head()

,node1,node2,node3
150,0.999990,0.999994,0.999993
150r,1.037486,1.037492,1.037490
149,1.037486,1.037492,1.037490
1,1.024960,1.035118,1.028590
2,NaN,1.034896,NaN


In [24]:
vmag_long = vmag_df.stack()
min_idx = vmag_long.idxmin()
min_bus, min_node = min_idx
min_v = vmag_long[min_idx]

print(f"Lowest voltage on the feeder")
print(f"  Bus     : {min_bus}")
print(f"  Node    : {min_node}")
print(f"  Voltage : {min_v:.4f} pu")

Lowest voltage on the feeder
  Bus     : 65
  Node    : node1
  Voltage : 0.9792 pu


## Add an energymeter and re-solve

Voltage profile and circuit plots in `py-dss-toolkit` rely on an energymeter so the toolkit knows where the feederhead is. The next cell uses a small toolkit helper that adds a line in series with the source plus an energymeter and a few monitors. Then we re-solve.

In [25]:
dss_tools.model.add_line_in_vsource(add_meter=True, add_monitors=True)
dss.text("solve")

''

## Voltage profile in one line

An interactive voltage profile of the entire feeder, color-coded by phase. This kind of plot is one of the things that took the most code before the toolkit existed.

In [26]:
dss_tools.interactive_view.voltage_profile(title="voltage profile | peak load")

## Circuit map colored by voltage

Same idea, different view — the whole feeder on a map, with voltage as the color. Useful for spotting where the low-voltage area actually sits geographically.

In [27]:
dss.text("buscoords BusCoords.dat")

''

In [28]:
dss_tools.interactive_view.voltage_settings.nodes_voltage_value = "min"
dss_tools.interactive_view.circuit_plot(
    parameter="voltage",
    title="voltage map | peak load",
)

## Side-by-side: with and without the toolkit

Same workflow, two ways to write it.

| Task | py-dss-interface | py-dss-toolkit |
|---|---|---|
| Read all line properties | manual loop over `dss.lines` | `dss_tools.model.lines_df` |
| Read all bus properties | manual loop over `dss.bus` | `dss_tools.model.buses_df` |
| Snapshot summary | manual call to `total_power`, `losses`, `buses_vmag_pu`, `min/max` | `dss_tools.results.summary_df` |
| Lowest voltage node | pair `nodes_names` and `buses_vmag_pu`, find min | `dss_tools.results.voltage_ln_nodes` + `stack()` + `idxmin()` |
| Voltage profile plot | build matplotlib or plotly by hand, or use DSSView.exe in Windows | `dss_tools.interactive_view.voltage_profile()` |
| Circuit map | build by hand or use DSSView.exe in Windows | `dss_tools.interactive_view.circuit_plot()` |

The toolkit is not a replacement. Underneath, it is calling the same `py-dss-interface` we used in Notebooks 1 to 3.

## Key takeaways

- `py-dss-interface` gives you direct, low-level control of OpenDSS from Python. That is the foundation.
- `py-dss-toolkit` builds on top of it with structured DataFrames, ready-made result tables, and interactive visualizations.
- The two work together. You always have `dss.text(...)` available — the toolkit just saves you from rewriting the same boilerplate.

This is the message of the workshop: OpenDSS is a powerful standalone simulator, but with `py-dss-interface` it becomes programmable, and with `py-dss-toolkit` the common engineering workflows become short and visual.

## Additional learning resources

If you would like to continue learning OpenDSS and Python for power-system studies, you can find more educational materials and courses here:

- [OpenDSS courses](https://www.pauloradatz.me/opendss-courses)

## Contact

For questions or follow-up about these materials:

- Paulo Radatz
- Email: [paulo.radatz@gmail.com](mailto:paulo.radatz@gmail.com)
- LinkedIn: [linkedin.com/in/pauloradatz](https://www.linkedin.com/in/pauloradatz/)
- Website: [pauloradatz.me](https://www.pauloradatz.me/)